# 🚀 LoRA 5-Fold Training
Notebook ini akan melatih LoRA untuk sisa Fold 1, 2, 3, dan 4 secara berurutan. Karena Fold 0 sudah selesai, kita akan langsung tancap gas dari Fold 1. Proses ini otomatis akan me- *save* bobot terbaik dari tiap Fold ke Google Drive.

Siapkan kopimu, dan biarkan GPU Colab bekerja keras semalaman!

In [2]:
import os, sys, shutil, glob, time
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Clone atau Pull Repo Terbaru
REPO_DIR = '/content/satria-data-bdcugm02'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# 3. Instalasi
!pip install -q -U "torchao>=0.16.0"
!pip install -q --no-warn-conflicts -r {REPO_DIR}/track_b/requirements.txt

Mounted at /content/drive
Already up to date.


In [3]:
# 4. Copy Cepat Gambar TRAIN ke Storage Lokal Colab (/tmp) agar I/O Super Cepat
DRIVE_TRAIN_DIR = '/content/drive/MyDrive/BDC2026/train'
LOCAL_TRAIN_DIR = '/tmp/dataset/train'

def copy_img_worker(args):
    src, dst = args
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
        shutil.copy2(src, dst)

if os.path.exists(DRIVE_TRAIN_DIR):
    print("🚀 Memulai copy cepat gambar TRAIN ke storage lokal Colab (/tmp)...")
    train_imgs = [f for f in glob.glob(os.path.join(DRIVE_TRAIN_DIR, "**", "*"), recursive=True) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    # Pertahankan struktur subfolder kelas
    files_to_copy = [(src, src.replace(DRIVE_TRAIN_DIR, LOCAL_TRAIN_DIR)) for src in train_imgs]

    with ThreadPoolExecutor(max_workers=32) as executor:
        list(executor.map(copy_img_worker, files_to_copy))
    print(f"✅ Selesai meng-copy {len(files_to_copy)} gambar Train ke {LOCAL_TRAIN_DIR}")
else:
    print(f"⚠️ Folder {DRIVE_TRAIN_DIR} tidak ditemukan!")


🚀 Memulai copy cepat gambar TRAIN ke storage lokal Colab (/tmp)...
✅ Selesai meng-copy 26527 gambar Train ke /tmp/dataset/train


In [4]:
# 5. Setup Script Training 5-Fold
sys.path.insert(0, os.path.join(REPO_DIR, 'track_a', 'src'))
sys.path.insert(0, os.path.join(REPO_DIR, 'track_b', 'src'))
sys.path.insert(0, os.path.join(REPO_DIR, 'track_b', 'experiments'))

import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler
import config
from config import CFG
from embed import load_encoder
from loaders import get_loaders_b
from losses_metrics import macro_f1
from seed_utils import set_seed
from lora_ft import build_variant, hf_processor_to_data_config, train_one_epoch
from dataclasses import replace

def run_train_fold(fold_id: int, variant: str, cfg, checkpoint: str, max_epochs: int = 5, n_last_blocks: int = 2):
    print(f"\n{'='*50}\n🔥 MEMULAI TRAINING FOLD {fold_id}\n{'='*50}")
    set_seed(cfg.seed)
    device = "cuda"

    if torch.cuda.is_available():
        import gc
        gc.collect()
        torch.cuda.empty_cache()

    revision = getattr(cfg, "backbone_revision", None)
    encoder, processor = load_encoder(checkpoint, device=device, revision=revision)
    hidden_size = getattr(getattr(encoder.config, "vision_config", encoder.config), "hidden_size", 1152)
    data_config = hf_processor_to_data_config(processor)

    model_img_size = data_config["input_size"][1]
    if cfg.img_size != model_img_size:
        cfg = replace(cfg, img_size=model_img_size)

    val_batch_size = getattr(cfg, "val_batch", 32)
    train_loader, val_loader, _ = get_loaders_b(fold=fold_id, cfg=cfg, data_config=data_config, val_batch_size=val_batch_size)

    model, optimizer = build_variant(variant, encoder, hidden_size, num_classes=cfg.num_classes, n_last_blocks=n_last_blocks)
    model = model.to(device)
    model = torch.compile(model)

    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()
    accum_steps = getattr(cfg, "accum_steps", 1)

    best_val_f1 = -1.0
    best_epoch = -1

    save_dir = getattr(cfg, "save_dir", "/content/drive/MyDrive/BDC2026apace/output_trackB")
    os.makedirs(save_dir, exist_ok=True)
    # Custom Run Name khusus untuk iterasi Fold
    run_name = f"{variant}_ft_fold{fold_id}_{max_epochs}ep_v3"
    ckpt_path = os.path.join(save_dir, f"{run_name}_best.pt")

    for epoch in range(max_epochs):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device, accum_steps=accum_steps)
        elapsed = time.time() - t0

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    outputs = model(images)
                all_preds.append(outputs.argmax(dim=1).cpu())
                all_labels.append(labels)
        val_f1 = macro_f1(torch.cat(all_preds), torch.cat(all_labels))

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch + 1
            # Auto-save
            trainable_keys = {k for k, v in model.named_parameters() if v.requires_grad}
            trainable_state = {k: v.cpu() for k, v in model.state_dict().items() if k in trainable_keys}
            torch.save(trainable_state, ckpt_path)
            print(f"  ✅ [SAVE] Epoch {best_epoch} -> F1: {val_f1:.4f} (Tersimpan ke Drive!)")

        print(f"  [Fold {fold_id}] Epoch {epoch+1}/{max_epochs} | Loss: {tr_loss:.4f} | Val F1: {val_f1:.4f} | Waktu: {elapsed/60:.1f} mnt")

    del model, encoder, optimizer, scaler
    torch.cuda.empty_cache()
    return best_val_f1


In [ ]:
# 6. EKSEKUSI TRAINING SISA FOLD (1, 2, 3, 4)
# Karena Fold 0 sudah selesai, kita gas dari Fold 1 sampai 4
VARIANT = "lora"
CHECKPOINT_HF = "google/siglip2-so400m-patch14-384"
MAX_EPOCHS = 5

# Pastikan kita pakai data folds_v3 (dataset yang udah bersih)
from dataclasses import replace
CFG = replace(CFG, folds_csv=os.path.join(CFG.save_dir, "..", "output_trackA", "folds_v3.csv"))
print(f"Menggunakan dataset: {CFG.folds_csv}")

results = {}
for fold in range(1, 5):
    best_f1 = run_train_fold(
        fold_id=fold,
        variant=VARIANT,
        cfg=CFG,
        checkpoint=CHECKPOINT_HF,
        max_epochs=MAX_EPOCHS
    )
    results[f"Fold_{fold}"] = best_f1

print("\n" + "="*50)
print("🎉🎉🎉 SEMUA FOLD SELESAI DILATIH! 🎉🎉🎉")
print("==================================================")
for k, v in results.items():
    print(f"{k}: Best F1 = {v:.4f}")


Menggunakan dataset: /content/drive/MyDrive/BDC2026apace/output_trackB/../output_trackA/folds_v3.csv

🔥 MEMULAI TRAINING FOLD 1
⚡ [Fast I/O] Path gambar dialihkan ke lokal SSD Colab: /tmp/dataset/train
[WasteDataset] Loaded 20658 sampel | dist: {0: 7782, 1: 3158, 2: 9718}
[WasteDataset] Loaded 5168 sampel | dist: {0: 1956, 1: 775, 2: 2437}


/tmp/ipykernel_21929/2483751847.py:49: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


  ✅ [SAVE] Epoch 1 -> F1: 0.9760 (Tersimpan ke Drive!)
  [Fold 1] Epoch 1/5 | Loss: 0.3810 | Val F1: 0.9760 | Waktu: 20.6 mnt
  ✅ [SAVE] Epoch 2 -> F1: 0.9833 (Tersimpan ke Drive!)
  [Fold 1] Epoch 2/5 | Loss: 0.0473 | Val F1: 0.9833 | Waktu: 20.7 mnt
  ✅ [SAVE] Epoch 3 -> F1: 0.9897 (Tersimpan ke Drive!)
  [Fold 1] Epoch 3/5 | Loss: 0.0320 | Val F1: 0.9897 | Waktu: 20.8 mnt
  ✅ [SAVE] Epoch 4 -> F1: 0.9932 (Tersimpan ke Drive!)
  [Fold 1] Epoch 4/5 | Loss: 0.0260 | Val F1: 0.9932 | Waktu: 20.7 mnt
  ✅ [SAVE] Epoch 5 -> F1: 0.9932 (Tersimpan ke Drive!)
  [Fold 1] Epoch 5/5 | Loss: 0.0230 | Val F1: 0.9932 | Waktu: 20.7 mnt

🔥 MEMULAI TRAINING FOLD 2
⚡ [Fast I/O] Path gambar dialihkan ke lokal SSD Colab: /tmp/dataset/train
[WasteDataset] Loaded 20660 sampel | dist: {0: 7791, 1: 3136, 2: 9733}
[WasteDataset] Loaded 5166 sampel | dist: {0: 1947, 1: 797, 2: 2422}


/tmp/ipykernel_21929/2483751847.py:49: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


  ✅ [SAVE] Epoch 1 -> F1: 0.9752 (Tersimpan ke Drive!)
  [Fold 2] Epoch 1/5 | Loss: 0.3820 | Val F1: 0.9752 | Waktu: 20.8 mnt
  ✅ [SAVE] Epoch 2 -> F1: 0.9869 (Tersimpan ke Drive!)
  [Fold 2] Epoch 2/5 | Loss: 0.0489 | Val F1: 0.9869 | Waktu: 20.8 mnt
  ✅ [SAVE] Epoch 3 -> F1: 0.9886 (Tersimpan ke Drive!)
  [Fold 2] Epoch 3/5 | Loss: 0.0325 | Val F1: 0.9886 | Waktu: 20.8 mnt
